# Multimodal text–image embeddings with vision language models

**Run in Google Colab only**.

**In one sentence:** turn text *and* images into comparable numbers in the **same vector space**, then score how well a text query matches an image — the first step toward “search photos with words.”

**What you will learn**

- Load a **vision language model (VLM)** embedding model ([Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B)) with [Sentence Transformers](https://www.sbert.net/)
- Encode **text** and **image** inputs into one shared vector space (both use `model.encode()`)
- Measure **cross-modal similarity** (text query vs image documents) and use `encode_query` / `encode_document` for retrieval-style APIs

**Notebook roadmap**

| Section | What happens | What to expect |
|---------|--------------|----------------|
| 1 | Colab + GPU setup | Fails fast if no GPU |
| 2 | Demo images, helpers, previews | Car + bee thumbnails |
| 3 | Load Qwen3-VL embedder | Slow first run (~4 GB download) |
| 4 | Encode 2 images | Shape `(2, 2048)` |
| 5 | **Cross-modal similarity** | Text rows × image columns; check ranking |
| 6 | Retrieval API | `encode_query` vs `encode_document` |
| Wrap-up | Recap + next steps | Where to go from here |

**How to open (only supported path)**

1. [Google Colab](https://colab.research.google.com/) → **File → Open notebook → GitHub**
2. Enter URL: `https://github.com/ysskrishna/awesome-llm-experiments/blob/main/experiments/multimodal-text-image-vl-embeddings/notebook.ipynb`
3. **Runtime → Change runtime type → T4 GPU** (required; model needs ~8 GB VRAM)
4. Run all cells **top to bottom**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ysskrishna/awesome-llm-experiments/blob/main/experiments/multimodal-text-image-vl-embeddings/notebook.ipynb)

**First run:** [`Qwen/Qwen3-VL-Embedding-2B`](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) downloads from Hugging Face on first load (multi‑GB; exact size depends on format). Cached under `~/.cache/huggingface/hub/` in the Colab VM. Expect several minutes for install + download + first encode on a T4.

## Concepts — read this before the code

### Glossary (beginner-friendly)

| Term | Plain meaning |
|------|----------------|
| **Embedding / encode** | Turn text or an image into a fixed list of numbers (a **vector**) |
| **Vector space** | All vectors live in the same space so you can compare them with one score |
| **Modality** | Input type: text, image, audio, … |
| **Cross-modal** | Compare **different** modalities (e.g. text ↔ image) |
| **Similarity** | How close two vectors are; higher score ≈ better semantic match |
| **Ranking** | For one text query, which image has the **highest** score? That image wins retrieval |
| **Modality gap** | Text–image scores are often lower than text–text; **order** still matters for search |

### The core idea (shared space)

Text and images are different raw data. The model maps **both** into the **same** 2048-dimensional space so you can compare them:

```
  "green car by building"  ──encode──►  [2048 numbers]  ─┐
  car photo URL            ──encode──►  [2048 numbers]  ─┼── same space → similarity()
  "bee on flower"          ──encode──►  [2048 numbers]  ─┤
  bee photo URL            ──encode──►  [2048 numbers]  ─┘
```

Close vectors ≈ similar meaning. Car caption should sit nearer the car image than the bee image.

```
Setup → Load model → encode(images) → encode(texts) → similarity matrix → check winner
                                              ↘ encode_query / encode_document ↗
```

### Quick links

| Idea | Link |
|------|------|
| Qwen3-VL embedding model | [Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) |
| Multimodal Sentence Transformers blog | [HF blog](https://huggingface.co/blog/multimodal-sentence-transformers) |
| Sentence Transformers API | [Documentation](https://www.sbert.net/) |
| Modality gap (lower cross-modal scores) | Section 5 below |

## 1. Colab environment — install dependencies and verify GPU

**What happens:** checks you are in Google Colab, installs libraries, and confirms a CUDA GPU is available.

**What to expect:** `GPU: Tesla T4` (or similar). Without a GPU the next cells will fail — the model needs ~8 GB VRAM.

Installs `sentence-transformers` with the **`[image]`** extra (v5.4+). That extra is required so `model.encode()` accepts image URLs/paths, not just strings.

In [1]:
# This notebook needs a Colab GPU (~8 GB VRAM). Local Jupyter is not supported.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    raise RuntimeError(
        "This notebook is Colab-only. Open it from GitHub in Google Colab "
        "(File → Open notebook → GitHub) and enable a GPU runtime."
    )

In [2]:
# [image] extra: model.encode() accepts image URLs/paths (multimodal API, v5.4+).
%pip install -q -U "sentence-transformers[image]>=5.4"

In [3]:
import torch

# Qwen3-VL-Embedding-2B runs on GPU; CPU is too slow and may OOM.
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not available. In Colab: Runtime → Change runtime type → "
        "Hardware accelerator → T4 GPU, then restart and run all cells."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


## 2. Define helpers and demo constants

**What happens:** sets the model name, two public demo images (car + bee), and small helpers to print and check similarity tables.

**What to expect:** no output from the code cell below — definitions only. The **next** cell shows image thumbnails. The model is **not** loaded yet.

**Demo corpus (our tiny “database”):**

| Index | Label | What it is |
|-------|-------|------------|
| 0 | `car` | Green car in front of a yellow building |
| 1 | `bee` | Bee on a pink flower |

Later, text captions will be scored against **both** images. Retrieval means: for each caption, pick the image column with the **highest** score.

**Next cell:** thumbnail preview of each image with its **index** (0 = car, 1 = bee) — the same labels used as column headers in the similarity tables.


In [4]:
# Vision-language model: maps text AND images into the same vector space.
MODEL_NAME = "Qwen/Qwen3-VL-Embedding-2B"

# Tiny demo corpus: two public images we will treat as "documents" to search over.
CAR_IMAGE = (
    "https://huggingface.co/datasets/huggingface/documentation-images/"
    "resolve/main/transformers/tasks/car.jpg"
)
BEE_IMAGE = (
    "https://huggingface.co/datasets/huggingface/documentation-images/"
    "resolve/main/bee.jpg"
)
DEMO_IMAGES = [CAR_IMAGE, BEE_IMAGE]  # index 0 = car, index 1 = bee
IMAGE_LABELS = ["car", "bee"]  # column names when printing similarity tables


def print_similarity_matrix(similarities, row_labels, col_labels):
    """Print text×image similarity scores with row/column labels."""
    # similarities[i, j] = how well text row i matches image column j (cosine-like score).
    sim = similarities.cpu()
    header = " " * 24 + "  ".join(f"{c:>8}" for c in col_labels)
    print(header)
    for i, row_name in enumerate(row_labels):
        scores = "  ".join(f"{sim[i, j].item():8.4f}" for j in range(sim.shape[1]))
        print(f"{row_name:24} {scores}")


def best_image_index(similarities, text_row: int, num_images: int) -> int:
    """Index of the image column with highest similarity for one text row."""
    # Retrieval rule: pick the image with the highest score for this text query.
    row = similarities[text_row, :num_images]
    return int(row.argmax().item())


def assert_text_prefers_image(similarities, text_row, expected_image_idx, caption):
    # We check ranking (correct image wins), not absolute score thresholds.
    best = best_image_index(similarities, text_row, similarities.shape[1])
    assert best == expected_image_idx, (
        f"{caption}: expected image index {expected_image_idx}, got {best}"
    )

In [5]:
from IPython.display import Image, Markdown, display

# Thumbnails for our tiny image "corpus" — same URLs passed to model.encode() later.
for idx, (label, url) in enumerate(zip(IMAGE_LABELS, DEMO_IMAGES)):
    display(Markdown(f"**Index `{idx}` — `{label}`** (column `{label}` in similarity tables)"))
    display(Image(url=url, width=360))

**Index `0` — `car`** (column `car` in similarity tables)

**Index `1` — `bee`** (column `bee` in similarity tables)

## 3. Load the vision language embedding model

**What happens:** downloads (first run only) and loads **Qwen3-VL-Embedding-2B** onto the GPU via Sentence Transformers.

**What to expect:** progress bars on first run (~4 GB). Then:

```
Modalities: ['text', 'image', 'video', 'message']
```

We use **text** and **image** only. This model is an *embedder* — it outputs vectors for search, not chat replies.

**After this cell you can call:**

- `model.encode(texts)` → text vectors
- `model.encode(image_urls)` → image vectors  
Both land in the **same** vector space (2048 dimensions for this model).

In [6]:
from sentence_transformers import SentenceTransformer

# First run downloads ~4 GB from Hugging Face and caches under ~/.cache/huggingface/hub/.
model = SentenceTransformer(MODEL_NAME)

# This experiment needs image encoding; text-only models would fail here.
assert model.supports("image"), "Expected image modality support"
print(f"Modalities: {model.modalities}")  # text, image, video, message — we use text + image

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/770 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

Modalities: ['text', 'image', 'video', 'message']


## 4. Encode images — expect shape `(2, 2048)`

**What happens:** each image URL is passed through the model once. Pixels → one vector of 2048 numbers. This is the **index-time** step in a real search system (store vectors, search later).

**What to expect:** `Image embeddings shape: (2, 2048)`

- **2** = two images (car, bee)
- **2048** = embedding dimension (fixed size per vector)

```
  car.jpg  ──model.encode()──►  row 0: [2048 floats]
  bee.jpg  ──model.encode()──►  row 1: [2048 floats]
         = img_embeddings matrix (2 × 2048)
```

In section 5 we encode **text** into the same kind of vectors and compare text rows to these image rows.

In [7]:
# encode() turns each image URL into a fixed-size vector (same space as text will use later).
# Shape: (num_images, embedding_dim) — here (2, 2048) for Qwen3-VL-Embedding-2B.
img_embeddings = model.encode(DEMO_IMAGES)
print(f"Image embeddings shape: {img_embeddings.shape}")
assert img_embeddings.shape[0] == len(DEMO_IMAGES)
assert img_embeddings.shape[1] > 0

Image embeddings shape: (2, 2048)


## 5. Cross-modal similarity — text queries vs image documents

**This is the main demo.** “Cross-modal” = compare **text** (rows) to **images** (columns) in the shared vector space.

### Step-by-step

![Cross-modal similarity steps](https://raw.githubusercontent.com/ysskrishna/awesome-llm-experiments/main/experiments/multimodal-text-image-vl-embeddings/diagrams/cross-modal-steps.png)

### The four text captions

| Label | Text | Expected winner |
|-------|------|-----------------|
| `car_match` | Green car parked in front of a yellow building | **car** (col 0) |
| `car_negative` | Red car driving on a highway | car-related (harder) |
| `bee_match` | Bee on a pink flower | **bee** (col 1) |
| `bee_negative` | Wasp on a wooden table | insect but not bee |

### How to read the printed table

```
                         car       bee
car_match              0.5117    0.1108   ← 0.51 > 0.11 → car wins ✓
bee_match              0.1237    0.6788   ← 0.68 > 0.12 → bee wins ✓
```

- **Rows** = text queries  
- **Columns** = image documents  
- **Higher number** = better match  
- We assert **ranking** (correct image wins), **not** “score must be above 0.9”

### Modality gap

Cross-modal scores (text vs image) are often **lower** than within-modal scores (text vs text). That is normal. Search still works when the **right** image scores higher than the wrong one.

**What to expect:** a labeled similarity table, then `Rank checks passed: car caption → car image; bee caption → bee image.`

In [8]:
# --- Cross-modal similarity demo ---
# "Cross-modal" = compare different input types (text vs image) in one shared vector space.
# Rows = text queries; columns = image documents. Higher score = better semantic match.

texts = [
    "A green car parked in front of a yellow building",  # should match car image (idx 0)
    "A red car driving on a highway",  # car-related but harder (different scene)
    "A bee on a pink flower",  # should match bee image (idx 1)
    "A wasp on a wooden table",  # insect-related hard negative for bee image
]
text_labels = ["car_match", "car_negative", "bee_match", "bee_negative"]

# Step 1: embed images and texts into the same vector space.
img_embeddings = model.encode(DEMO_IMAGES)  # shape (2, 2048)
text_embeddings = model.encode(texts)  # shape (4, 2048)

# Step 2: pairwise scores — similarities[i, j] = text i vs image j.
# This is the core "search photos with words" operation (no vector DB yet).
similarities = model.similarity(text_embeddings, img_embeddings)

print_similarity_matrix(similarities, text_labels, IMAGE_LABELS)

# Step 3: verify retrieval ranking, not raw score magnitude (modality gap lowers scores).
assert_text_prefers_image(similarities, 0, 0, "Green car caption")
assert_text_prefers_image(similarities, 2, 1, "Bee on flower caption")
print("Rank checks passed: car caption → car image; bee caption → bee image.")

                             car       bee
car_match                  0.5117    0.1108
car_negative               0.2008    0.1139
bee_match                  0.1237    0.6788
bee_negative               0.1280    0.2727
Rank checks passed: car caption → car image; bee caption → bee image.


## 6. Retrieval-style API — `encode_query` and `encode_document`

**What happens:** same embed → compare → rank idea as section 5, but using the **search / index** API many production embedders expose.

| When | Function | Role |
|------|----------|------|
| **Index time** (build library once) | `encode_document(images)` | “These items are in my corpus” |
| **Search time** (each user query) | `encode_query(text)` | “This is what I’m looking for” |

Both still produce vectors in the **same space**. The names differ because the model may apply different internal prompts for queries vs documents (better retrieval quality).

![encode_query / encode_document sequence](https://raw.githubusercontent.com/ysskrishna/awesome-llm-experiments/main/experiments/multimodal-text-image-vl-embeddings/diagrams/encode-query-sequence.png)

In a full search pipeline you would store `encode_document` vectors in a vector DB and run `encode_query` at search time.

**What to expect:** store `encode_document` vectors in a vector DB; at search time run `encode_query` and fetch nearest neighbors.

**What to expect:** another similarity table and `Retrieval rank checks passed.`

In [ ]:
# --- Retrieval-style API (closer to production search) ---
# Many embedders use different internal prompts for "I am searching" vs "I am being searched".
# encode_query = search side; encode_document = index side (images stored in a vector DB).

queries = [
    "Find me a photo of a vehicle parked near a building",  # expect car (idx 0)
    "Show me an image of a pollinating insect",  # expect bee (idx 1)
]

query_embeddings = model.encode_query(queries)  # user text at search time
doc_embeddings = model.encode_document(DEMO_IMAGES)  # images at index time
retrieval_sims = model.similarity(query_embeddings, doc_embeddings)

print_similarity_matrix(
    retrieval_sims,
    ["vehicle_query", "insect_query"],
    IMAGE_LABELS,
)

# Same idea as section 5: correct document should rank first for each query.
assert best_image_index(retrieval_sims, 0, 2) == 0, "Vehicle query should rank car image first"
assert best_image_index(retrieval_sims, 1, 2) == 1, "Insect query should rank bee image first"
print("Retrieval rank checks passed.")

                             car       bee
vehicle_query              0.3947    0.1515
insect_query               0.1266    0.4861
Retrieval rank checks passed.


## Wrap-up

### What you did

1. Loaded a **multimodal VLM embedder** (Qwen3-VL-Embedding-2B)
2. Encoded **images** and **text** into the **same** 2048-dim vector space
3. Built a **text × image similarity matrix** and checked that the right image **ranks first**
4. Repeated with **`encode_query`** / **`encode_document`** (retrieval-style API)

### The story in one line

**Search photos with words** = embed query text + embed image corpus → compare vectors → return highest-scoring images.

![This notebook vs production next steps](https://raw.githubusercontent.com/ysskrishna/awesome-llm-experiments/main/experiments/multimodal-text-image-vl-embeddings/diagrams/wrap-up-next-steps.png)

### If you hit OOM on Colab

Reload with lower memory, e.g. `SentenceTransformer(MODEL_NAME, model_kwargs={"torch_dtype": "bfloat16"})` — see the [blog’s processor/model kwargs section](https://huggingface.co/blog/multimodal-sentence-transformers#processor-and-model-kwargs).